# étoiles Wolf-Rayet
$\rightarrow$ **analyse de l'étoile v1770 cyg**

## la cible

- V1770 Cyg = WR 136 est une étoile Wolf-Rayet massive en fin de vie, au cœur de NGC 6888. 
- elle est classée WN6 (Wolf-Rayet de type azote)
- son spectre montre des raies en emission :
    - He II λ4686 Å,
    - multiplets N III λ4640–4642 Å
    - N IV λ4058 Å
    - raies larges (plusieurs dizaines d’Å) $\rightarrow$ vitesses de vent stellaire de 1000–2000 km/s.

**attention** : les raies de la série de Balmer peuvent être confondues avec des transitions de He II, ce qui complique l’identification.

## les données
|||
|---|---|
|OBJECT|	v1770 cyg|
|EXPTIME2|	5 x 60 s|
|BSS_ITRP|	1500|
|SPE_RPOW|	1500|
|BSS_VHEL|	0|
|DATE-OBS|	2025-11-02T21:44:08.321|
|BSS_SITE|	CALC|
|BSS_INST|	C11 + Dados200 + 25mic + PO_UranusM|
|||




## spectro dashboard
- lancer la cellule suivante
- sur 'Colormap', bouton droit : "create new view for cell output"
- redimensionner ou déplacer l'onglet créé 'Output View' 

In [2]:
%matplotlib widget
import numpy as np
from spectra_widget import SpectraWidget

# 1. Afficher le dashboard
db = SpectraWidget()
db.show()


In [3]:
# import de base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
# import astropy et autre affiliated libs
import astropy.units as u
from astropy.io import fits
from astropy.nddata import StdDevUncertainty
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.time import Time
from astropy.modeling import models
from astropy.stats import mad_std

from specutils import Spectrum
from specutils.fitting import fit_generic_continuum, fit_continuum
from specutils.analysis import centroid, fwhm, snr, snr_derived
from specutils.spectra import SpectralRegion
from specutils.manipulation import extract_region



In [5]:

# Charger le spectre
db.clear_spectra()

spec_name = "data/plouis/_v1770cyg_20251102_906.fits"
spec = Spectrum.read(spec_name)


# montre le header FITS
#header = spec.meta['header']
#df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])
#display(df.style.hide(axis='index'))   

db.show_spectrum(spec.spectral_axis, spec.flux, label=spec.meta['header']['OBJECT'])


INFO: affichage du spectre 'v1770 cyg' : 6346 pts, X:[3700.3:7349.2]


In [6]:
# soustraction du continuum
_fit = fit_continuum(spec, model=models.Polynomial1D(degree=2))

continuum = _fit(spec.spectral_axis)
spec_norm = spec / continuum

db.show_spectrum(spec.spectral_axis, spec_norm.flux, label=f"{spec.meta['header']['OBJECT']} - sans continnuum")
#db.show_spectrum(spec.spectral_axis, continuum, label=f"continnuum")


INFO: affichage du spectre 'v1770 cyg - sans continnuum' : 6346 pts, X:[3700.3:7349.2]


In [7]:
# les raies sélectionnées :

lines_ref = pd.DataFrame({
    "elem": [
            "N IV", 
            "He II", 
            "N III", 
            #"He II",
            "He II",
            "He II", 
            "C IV", 
            "He I",
            "He II/Hα", 
            "N IV",
    ],

    "lambda_rest": [
            4058.0, 
            4340.0, 
            4640.0, 
            #4686.0,
            4859.0,
            5411.0, 
            5800.6, 
            5875.6,
            6562.8, 
            7109.0, 
    ]
})


In [9]:
from specutils.spectra import SpectralRegion
from specutils.manipulation import extract_region
from specutils.analysis import centroid, fwhm, snr_derived
import numpy as np
import pandas as pd
import astropy.units as u

# DADOS 200 : R ~ 1350 → Δλ ~ 4.6 Å → Δv ~ 100 km/s

c_kms = 299792.458 * u.km / u.s
region_width = 25

# Calcul des erreurs systématiques (DADOS 200)
fwhm_neon_pix = 8   #px
delta_lambda = np.mean(np.diff(spec_norm.spectral_axis)) # Dispersion (A/pix)

fwhm_inst_angstrom = fwhm_neon_pix * delta_lambda

region_cont = SpectralRegion(5000.0 * u.AA, 5300.0 * u.AA) 
sub_spectrum = extract_region(spec, region_cont)
flux_zone = sub_spectrum.flux.value
#snr_val = np.median(flux_zone) / mad_std(flux_zone)
snr_val = snr_derived(sub_spectrum) #.value


print (f"{delta_lambda=:.4f}", f"{snr_val=:.0f}")

results = []
for _, row in lines_ref.iterrows():
    lam_rest = row["lambda_rest"] * u.AA
    elem = row["elem"]
    region = SpectralRegion(lam_rest - region_width * u.AA, lam_rest + region_width * u.AA)
    subspec = extract_region(spec_norm, region)

    # Mesures
#    cen = centroid(subspec)
    fwhm_brut = fwhm(subspec)
    v_expansion = (c_kms * (fwhm_brut / lam_rest))

    # Incertitudes stats
    err_fwhm_stat = fwhm_brut.value / snr_val       
    err_v_expansion = (c_kms * (err_fwhm_stat / lam_rest))

    # Incertitudes syst
    sigma_syst_rv = (fwhm_inst_angstrom / 10.0 / lam_rest) * c_kms # Règle du 1/10eme
    
    # incertitudes RMS (statistiques + systématiques)
    err_v_exp_total = np.sqrt(err_v_expansion.value**2 + sigma_syst_rv.value**2)

    results.append({
        "Élément": elem,
        "Lambda (Å)": f"{lam_rest.value}",
        "FWHM (Å)": f"{fwhm_brut.value:.2f} ± {err_fwhm_stat:.2f}",
        "V expansion (km/s)": f"{v_expansion.value / 2:.0f} ± {err_v_expansion.value:.0f}",
    })

df = pd.DataFrame(results)
df


delta_lambda=0.5751 Angstrom snr_val=155


,Élément,Lambda (Å),FWHM (Å),V expansion (km/s)
0,N IV,4058.0,49.46 ± 0.32,1827 ± 24
1,He II,4340.0,49.46 ± 0.32,1708 ± 22
2,N III,4640.0,49.46 ± 0.32,1598 ± 21
3,He II,4859.0,44.65 ± 0.29,1377 ± 18
4,He II,5411.0,46.16 ± 0.30,1279 ± 16
5,C IV,5800.6,49.46 ± 0.32,1278 ± 16
6,He I,5875.6,49.46 ± 0.32,1262 ± 16
7,He II/Hα,6562.8,49.46 ± 0.32,1130 ± 15
8,N IV,7109.0,49.46 ± 0.32,1043 ± 13


## analyse

Les raies des éléments plus ionisés (N IV, He II) ou de plus courte longueur d'onde ont des vitesses d'expansion plus élevées (autour de $1700-1800\ \text{km/s}$) que celles des raies de plus longue longueur d'onde (autour de $1100-1200\ \text{km/s}$).

-> les zones internes et plus chaudes (où se forment N IV et He II) s'éjectent plus rapidement que les zones externes ou plus froides.

## vu des pros
- Le spectre est dominé par le doublet He / N à 4640-86 Å --> type WN6.
- En raison de la température élevée de ces étoiles, des raies d'hélium ionisé sont observées avec du carbone et de l'azote.
- Les lignes d'hélium ionisées ont la même position que les lignes d'hydrogène **divisées par quatre**, en raison de la charge + 2 dans le noyau He+.
- Les raies qui semblent dues aux raies de Balmer dans les spectres WR sont en fait dues à la série de Brackett (n1 = 4 et n2 = 6, 7, 8, 9) de He+.
- He+ peut être distingué de l'hydrogène de la raie 4685Å qui appartient à la série de Paschen (n1 = 3, n2 = 4) et ne chevauche aucune longueur d'onde d'hydrogène existante.

